# Explainable Machine Learning for Hemoglobin-Independent Anemia Severity Prediction using LightGBM

This notebook contains the experimental implementation for anemia severity prediction using hemoglobin-independent hematological features, comparative evaluation of tree-based machine learning models, and SHAP-based explainability.

## 1. Data Loading & Clinically Grounded Label Creation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

DATA_PATH = 'data/BDCBC7196_Hematology_Dataset.csv'
df = pd.read_csv(DATA_PATH)

print("Initial Shape:", df.shape)

df = df.dropna().reset_index(drop=True)
print("After NA Removal:", df.shape)

# Create anemia severity labels based on WHO 2024 hemoglobin thresholds.
# Reference:
# World Health Organization (2024). Guideline on haemoglobin cutoffs
# to define anaemia in individuals and populations.

def severity_group(row):
    """
    WHO 2024 anemia severity thresholds (g/dL)
    based on age and gender stratification.
    """
    hb = row['Hb']
    gender = str(row['Gender']).upper().strip()
    age = row['Age']

    # Children 6-23 months
    if 0.5 <= age < 2:
        if hb >= 10.5:
            return "Normal"
        elif hb >= 9.5:
            return "Mild"
        elif hb >= 7.0:
            return "Moderate"
        else:
            return "Severe"

    # Children 24-59 months
    elif 2 <= age < 5:
        if hb >= 11.0:
            return "Normal"
        elif hb >= 10.0:
            return "Mild"
        elif hb >= 7.0:
            return "Moderate"
        else:
            return "Severe"

    # Children 5-11 years
    elif 5 <= age < 12:
        if hb >= 11.5:
            return "Normal"
        elif hb >= 11.0:
            return "Mild"
        elif hb >= 8.0:
            return "Moderate"
        else:
            return "Severe"

    # Children 12-14 years
    elif 12 <= age < 15:
        if hb >= 12.0:
            return "Normal"
        elif hb >= 11.0:
            return "Mild"
        elif hb >= 8.0:
            return "Moderate"
        else:
            return "Severe"

    # Adults (>=15 years)
    elif age >= 15:
        # Women (non-pregnant)
        if gender in ['F', 'FEMALE', 'FEM']:
            if hb >= 12.0:
                return "Normal"
            elif hb >= 11.0:
                return "Mild"
            elif hb >= 8.0:
                return "Moderate"
            else:
                return "Severe"

        # Men
        elif gender in ['M', 'MALE', 'MAL']:
            if hb >= 13.0:
                return "Normal"
            elif hb >= 11.0:
                return "Mild"
            elif hb >= 8.0:
                return "Moderate"
            else:
                return "Severe"

    # Fallback
    return "Normal" if hb >= 12.0 else "Mild" if hb >= 11.0 else "Moderate" if hb >= 8.0 else "Severe"


df["Severity"] = df.apply(severity_group, axis=1)

# Encode categorical variables
le_gender = LabelEncoder()
df["Gender"] = le_gender.fit_transform(df["Gender"])

le_severity = LabelEncoder()
df["Severity_Encoded"] = le_severity.fit_transform(df["Severity"])

print(df["Severity"].value_counts())


## 2. Feature Selection

In [ ]:
from sklearn.preprocessing import StandardScaler

# Hemoglobin is intentionally excluded to make the prediction
# hemoglobin-independent.

FEATURES = [
    'Gender', 'Age', 'RBC', 'WBC', 'PLATELETS', 'LYMP', 'MONO', 'HCT',
    'MCV', 'MCH', 'MCHC', 'RDW', 'PDW', 'MPV', 'PCT'
]

X = df[FEATURES]
y = df["Severity_Encoded"]

X_scaled = X.copy()

print("Features Used:", len(FEATURES))


## 3. Model Training & 5-Fold Stratified Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight

y = df["Severity_Encoded"].astype(int)

classes = np.unique(y)
weights = compute_class_weight("balanced", classes=classes, y=y)
class_weights = dict(zip(classes, weights))

MODELS = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_split=20,
        min_samples_leaf=15,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=5,
        reg_lambda=15,
        eval_metric='mlogloss',
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        max_depth=3,
        num_leaves=12,
        learning_rate=0.035,
        min_child_samples=45,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=4,
        reg_lambda=12,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
cv_predictions = {}

print("=" * 60)
print("MODEL PERFORMANCE (5-FOLD CROSS-VALIDATION)")
print("=" * 60)

for name, model in MODELS.items():

    if name == "XGBoost":
        y_pred = cross_val_predict(
            model,
            X_scaled,
            y,
            cv=skf,
            method="predict"
        )
    else:
        y_pred = cross_val_predict(
            model,
            X_scaled,
            y,
            cv=skf
        )

    cv_predictions[name] = y_pred

    results[name] = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, average="macro"),
        "Recall": recall_score(y, y_pred, average="macro"),
        "Macro_F1": f1_score(y, y_pred, average="macro"),
        "Confusion": confusion_matrix(y, y_pred)
    }

    print(f"\n{name}:")
    print(f"  Accuracy:   {results[name]['Accuracy']:.4f}")
    print(f"  Precision:  {results[name]['Precision']:.4f}")
    print(f"  Recall:     {results[name]['Recall']:.4f}")
    print(f"  Macro F1:   {results[name]['Macro_F1']:.4f}")

print("\n" + "=" * 60)


## 4. Performance Evaluation Tables

In [ ]:
from sklearn.metrics import classification_report

# Select the model with the highest Macro F1 score.
best_model_name = max(results, key=lambda k: results[k]["Macro_F1"])
print(f"Best model: {best_model_name}")

y_best = cv_predictions[best_model_name]

correct_order = ['Normal', 'Mild', 'Moderate', 'Severe']
current_order = list(le_severity.classes_)
idx = [current_order.index(c) for c in correct_order]

# Table I: Dataset and feature summary
table1 = pd.DataFrame({
    "Metric": ["Total Samples", "Total Features", "Total Classes"],
    "Value": [len(df), len(FEATURES), len(le_severity.classes_)]
})
display(table1)

# Table II: Model performance
table2 = pd.DataFrame(results).T[
    ["Accuracy", "Precision", "Recall", "Macro_F1"]
]
display(table2)

# Table III: Best model class-wise performance
report = classification_report(
    y,
    y_best,
    target_names=le_severity.classes_,
    output_dict=True
)

table3_data = []

for class_name in correct_order:
    table3_data.append({
        "precision": report[class_name]['precision'],
        "recall": report[class_name]['recall'],
        "f1-score": report[class_name]['f1-score']
    })

table3 = pd.DataFrame(table3_data, index=correct_order)
display(table3)

# Table IV: Best model confusion matrix
cm_original = results[best_model_name]["Confusion"]
cm_reordered = cm_original[np.ix_(idx, idx)]

table4 = pd.DataFrame(
    cm_reordered,
    index=correct_order,
    columns=correct_order
)
display(table4)


## 5. Confusion Matrices

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correct_order = ['Normal', 'Mild', 'Moderate', 'Severe']
current_order = list(le_severity.classes_)
idx = [current_order.index(c) for c in correct_order]

for name in MODELS.keys():
    cm = results[name]["Confusion"]
    cm_reordered = cm[np.ix_(idx, idx)]

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_reordered,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=correct_order,
        yticklabels=correct_order
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{name} Confusion Matrix")
    plt.show()


## 6. Ablation Study — HCT Removed

In [ ]:
FEATURES_no_HCT = [f for f in FEATURES if f != "HCT"]
X_no_HCT = df[FEATURES_no_HCT]

X_scaled_no_HCT = X_no_HCT.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_no_HCT = {}
cv_predictions_no_HCT = {}

for name, model in MODELS.items():

    if name == "XGBoost":
        y_pred = cross_val_predict(
            model,
            X_scaled_no_HCT,
            y,
            cv=skf,
            method="predict"
        )
    else:
        y_pred = cross_val_predict(
            model,
            X_scaled_no_HCT,
            y,
            cv=skf
        )

    cv_predictions_no_HCT[name] = y_pred

    results_no_HCT[name] = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, average="macro"),
        "Recall": recall_score(y, y_pred, average="macro"),
        "Macro_F1": f1_score(y, y_pred, average="macro"),
        "Confusion": confusion_matrix(y, y_pred)
    }

# Display ablation metrics
table_ablation = pd.DataFrame(results_no_HCT).T[
    ["Accuracy", "Precision", "Recall", "Macro_F1"]
]

print("Ablation Study Metrics (HCT Removed)")
display(table_ablation)

# Plot confusion matrices
correct_order = ['Normal', 'Mild', 'Moderate', 'Severe']
current_order = list(le_severity.classes_)
idx = [current_order.index(c) for c in correct_order]

for name in MODELS.keys():
    cm = results_no_HCT[name]["Confusion"]
    cm_reordered = cm[np.ix_(idx, idx)]

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_reordered,
        annot=True,
        fmt='d',
        cmap='Oranges',
        xticklabels=correct_order,
        yticklabels=correct_order
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{name} Confusion Matrix (HCT Removed)")
    plt.show()


## 7. SHAP Explainability

In [ ]:
!pip install shap --quiet

import shap

# Get best model
best_model_obj = MODELS[best_model_name]

print("=" * 60)
print(f"SHAP ANALYSIS: {best_model_name}")
print("=" * 60)

# Train the selected model on the full dataset for explainability analysis.
best_model_obj.fit(X_scaled, y)

# Compute SHAP values
explainer = shap.TreeExplainer(best_model_obj)
shap_values = explainer.shap_values(X_scaled)

n_samples, n_features, n_classes = shap_values.shape
class_names = le_severity.classes_
severe_idx = list(class_names).index('Severe')

# Feature names corresponding to the original FEATURES order.
feature_names_clean = [
    'Gender',
    'Age (years)',
    'RBC (10^12/L)',
    'WBC (10^9/L)',
    'Platelets (10^9/L)',
    'Lymphocytes (%)',
    'Monocytes (%)',
    'Hematocrit (%)',
    'MCV (fL)',
    'MCH (pg)',
    'MCHC (g/dL)',
    'RDW (%)',
    'PDW (fL)',
    'MPV (fL)',
    'Plateletcrit (%)'
]

print("\nFeature order verification:")
for i, (orig, clean) in enumerate(zip(FEATURES, feature_names_clean)):
    print(f"  {i}: {orig} -> {clean}")


### Figure 1: Global SHAP Feature Importance

In [ ]:
global_importance = np.abs(shap_values).mean(axis=0).mean(axis=1)
global_importance_pct = (global_importance / global_importance.sum()) * 100

importance_df = pd.DataFrame({
    'Feature': feature_names_clean,
    'Importance (%)': global_importance_pct
}).sort_values('Importance (%)', ascending=True)

print("Top 5 Features:")
print(
    importance_df
    .sort_values('Importance (%)', ascending=False)
    .head(5)
)

fig1, ax1 = plt.subplots(figsize=(10, 6))

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(importance_df)))
bars = ax1.barh(
    importance_df['Feature'],
    importance_df['Importance (%)'],
    color=colors
)

for bar, val in zip(bars, importance_df['Importance (%)']):
    if val > 5:
        ax1.text(
            bar.get_width() - 1.5,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%',
            va='center',
            ha='right',
            fontsize=9,
            color='white',
            fontweight='bold'
        )
    else:
        ax1.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%',
            va='center',
            fontsize=9,
            fontweight='bold'
        )

ax1.set_xlabel('Relative Importance (%)', fontweight='bold', fontsize=12)
ax1.set_ylabel('', fontsize=12)
ax1.set_title('Global SHAP Feature Importance', fontweight='bold', fontsize=13)
ax1.set_xlim(0, max(importance_df['Importance (%)']) + 5)

plt.tight_layout()
plt.savefig('shap_fig1_global.png', dpi=300, bbox_inches='tight')
plt.show()


### Figure 2: SHAP Summary for Severe Class

In [ ]:
severe_shap = shap_values[:, :, severe_idx]

max_abs = np.abs(severe_shap).max()
severe_shap_normalized = severe_shap / max_abs * 0.5

fig2, ax2 = plt.subplots(figsize=(10, 6))

shap.summary_plot(
    severe_shap_normalized,
    X_scaled,
    feature_names=feature_names_clean,
    max_display=10,
    show=False,
    alpha=0.6
)

ax2 = plt.gca()
ax2.set_title(
    'SHAP Feature Impact - Severe Anemia Class',
    fontweight='bold',
    fontsize=13
)
ax2.set_xlabel('SHAP Value', fontweight='bold', fontsize=12)
ax2.set_ylabel('', fontsize=12)
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax2.set_xlim(-0.6, 0.6)

plt.tight_layout()
plt.savefig('shap_fig2_severe.png', dpi=300, bbox_inches='tight')
plt.show()


## 8. Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes = np.array([
    0.05, 0.1, 0.2, 0.3, 0.4,
    0.5, 0.6, 0.7, 0.8
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, model) in zip(axes, MODELS.items()):

    print(f"Computing learning curve for {name}...")

    train_sizes_abs, train_scores, valid_scores = learning_curve(
        model,
        X_scaled,
        y,
        train_sizes=train_sizes,
        cv=skf,
        scoring='f1_macro',
        n_jobs=-1,
        shuffle=True,
        random_state=42
    )

    train_mean = train_scores.mean(axis=1)
    valid_mean = valid_scores.mean(axis=1)

    ax.plot(
        train_sizes_abs,
        train_mean,
        'o-',
        color='blue',
        linewidth=2,
        label='Training'
    )
    ax.plot(
        train_sizes_abs,
        valid_mean,
        'o-',
        color='orange',
        linewidth=2,
        label='Validation'
    )

    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Training Samples')
    ax.set_ylabel('Macro F1-Score', fontweight='bold')
    ax.set_ylim(0.3, 0.95)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

    ax.text(
        0.65,
        0.25,
        f'Final: {valid_mean[-1]:.3f}',
        transform=ax.transAxes,
        fontsize=10,
        fontweight='bold'
    )

plt.suptitle(
    'Learning Curves for Hemoglobin-Independent Anemia Severity Prediction',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

print("Learning curves use Macro F1.")


## 9. Ordinal Analysis & Statistical Testing

In [ ]:
from scipy.stats import friedmanchisquare

# Ordinal error distribution
ordinal_diff = cv_predictions[best_model_name] - y
diff_counts = pd.Series(ordinal_diff).value_counts().sort_index()

diff_counts.plot(kind="bar")
plt.title("Ordinal Misclassification Distribution")
plt.show()

# Friedman test
rf = cv_predictions["RandomForest"]
xgb = cv_predictions["XGBoost"]
lgb = cv_predictions["LightGBM"]

stat, p = friedmanchisquare(rf, xgb, lgb)

print("Friedman test p-value:", p)
